# Reference-based selection of a loss--link specification

This notebook implements the design in `notebooks/experiments/REFERENCE_SELECTION_PLAN.md`. Each section answers one question a referee asks about the selection theorems in the manuscript.

| Section | Experiment | Claim under test |
|---|---|---|
| 2 | E1a | Raw Bregman objectives cannot rank candidates across generators |
| 4 | E1b | The proposed criterion beats the rules a practitioner would otherwise use |
| 5 | E2 | The candidate bias bound is valid *and* not vacuous |
| 6 | E3 | The oracle inequality's remainder is informative |
| 7 | E4 | Coverage is uniform over the calibrated bias family |
| 8 | E5 | What happens when the reference allowance is wrong |
| 9 | E6 | Bias awareness costs little when the bias is negligible |

Set `TIER` below. `smoke` runs in under a minute, `pilot` in about an hour, and `publication` is the full run reported in the manuscript (roughly 99 core-hours; set `MAX_WORKERS`). Only the replication count changes across tiers: the candidate library, selection rules, designs, and seed construction are identical.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, *REPO_ROOT.parents):
    if (candidate / "src" / "genriesz").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Run the notebook from inside the genriesz repository.")

for path in (REPO_ROOT / "src", REPO_ROOT / "notebooks" / "experiments"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from refsel import grids, report, rescaling  # noqa: E402
from refsel.runner import load_experiment, run_experiment  # noqa: E402

TIER = "smoke"
MAX_WORKERS = None

OUTPUT_ROOT = REPO_ROOT / "notebooks" / "experiments" / "results" / "reference_selection"
RUN_DIR = OUTPUT_ROOT / TIER
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save(table: pd.DataFrame, name: str) -> pd.DataFrame:
    """Display a table and write the CSV that the manuscript reads."""
    if table.empty:
        print(f"{name}: no rows at this tier")
        return table
    display(table)
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)
    return table

## 2. E1a: a raw Bregman objective is not a ranking

Replacing a generator $g$ by $\kappa g$ leaves the unpenalized fitted representer unchanged and multiplies both the objective value and the held-out Bregman criterion by $\kappa$. The estimator is identical; its cross-validation score is arbitrary. This is deterministic, so one table settles it and no Monte Carlo is needed.

Read `alpha_max_deviation` (should sit at the solver tolerance) against `objective_ratio` and `heldout_ratio` (should equal $\kappa$).

The `penalty` column closes the obvious escape route. Substituting $\beta = \kappa b$ shows the invariance survives whenever the penalty is positively homogeneous of degree one, and the $\ell_1$ penalty used throughout is. So the incomparability is **not** an artefact of dropping regularization: it holds for the penalized estimator actually fitted. Under $\ell_2$ the fitted representer does move, but only because rescaling the generator silently changes the effective penalty, which is no more defensible a basis for ranking.

In [ ]:
save(rescaling.rescaling_table(), "e1a_rescaling")

## 3. Run the experiment

The hidden-direction scales come from `refsel/calibration.json`, which is committed so that publication runs are deterministic and never re-calibrate. Completed batches are skipped, so an interrupted run resumes without changing its random-number allocation.

In [ ]:
config = grids.experiment_config(TIER, max_workers=MAX_WORKERS)
display(pd.DataFrame([scenario.__dict__ for scenario in config.scenarios]))
print(
    f"{len(config.scenarios)} scenarios, "
    f"{sum(config.replications(s) for s in config.scenarios)} replication jobs"
)

run_experiment(config, RUN_DIR)
tables = load_experiment(RUN_DIR)
{name: frame.shape for name, frame in tables.items()}

## 4. E1b: the selection horse race

Every rule below chooses from the *same* fitted library on the *same* fold, so the comparison is paired and the extra rules cost no additional fitting.

- `bregman_cv` is the naive rule that section 2 shows to be arbitrary.
- `lsif_cv` is the strong competitor: a generator-agnostic squared risk, comparable across candidates but measuring the error in $\alpha$ rather than drift in the target parameter.
- `abs_drift` drops the simultaneous radius, isolating what $q_a$ buys.
- `fixed_*` are the specifications a practitioner would pick without any selection.
- `oracle` is infeasible and bounds what any rule could achieve.

Coverage is unconditional: a rule that produced no estimate counts as a non-covering replication. `fixed_bkl` is expected to be unavailable throughout, because the bounded Kullback--Leibler generator leaves its domain on every ATE fold. That is a property of the estimand, not a solver failure, and the admissibility screen is supposed to catch it.

In [ ]:
save(report.selection_rule_table(tables), "e1b_selection_rules")

In [ ]:
save(report.selection_frequency_table(tables), "e1b_selection_frequency")
save(report.failure_table(tables), "numerical_failures")

## 5. E2: is the bias bound valid, and is it tight?

`lower_coverage` is the share of candidates with $|B_a| \le \widehat U_a$; `upper_coverage` checks the other half of Theorem `data_dependent_bias`, $\widehat U_a \le |B_a| + 2(q_a + b_r)$. Reporting only the first cannot distinguish a valid bound from a vacuous one.

`radius_share` and `allowance_share` decompose $\widehat U_a$ and say which term binds. `reference_drift` and `allowance_covers_reference` record whether the theorem's premise $|B_r| \le b_r$ actually held, rather than assuming it.

In [ ]:
save(report.bias_bound_table(tables), "e2_bias_bounds")

## 6. E3: the oracle inequality

`risk_ratio` is the realized conditional risk of the selected candidate divided by that of the best admissible candidate. Corollary `oracle_remainder` predicts it converges to one when the diagnostic error and the allowance are small relative to the oracle risk.

In [ ]:
save(report.oracle_regret_table(tables), "e3_oracle_regret")

## 7. E4: uniform coverage over the calibrated bias family

This is the centrepiece. The hidden-direction scale is calibrated so that a fixed benchmark specification attains a target bias-to-standard-error ratio $t$, the parameter indexing the bounded-normal-mean problem. Calibrating against a fixed benchmark keeps the definition of the family independent of the selection rule.

**Read the `min` rows.** A uniform-coverage claim is a statement about the worst case in the family, not the average. The ordinary Wald interval should decay along the dotted theoretical curve $2\Phi(1.96 - t) - 1$, while the bias-aware and conservative intervals should hold across the sweep.

In [ ]:
uniform = report.uniform_coverage_table(tables)
save(uniform, "e4_uniform_coverage")

# The headline row: the least favourable point of the family, with the standard
# error and interval length taken from that same scenario rather than from a
# column-wise minimum that would mix scenarios.
save(report.worst_case_coverage_table(tables), "e4_worst_case_coverage")

In [ ]:
sweep = uniform.copy()
if not sweep.empty:
    for sample_size, frame in sweep.groupby("sample_size"):
        frame = frame.sort_values("target_t")
        figure, axis = plt.subplots(figsize=(7.0, 4.5))
        for label, column in (
            ("Ordinary Wald", "wald_split_coverage"),
            ("Bias-aware (single split)", "bias_aware_split_coverage"),
            ("Conservative cross-fitted", "conservative_cf_coverage"),
        ):
            if column not in frame:
                continue
            error = frame.get(f"{column}_mcse")
            axis.errorbar(
                frame["target_t"],
                frame[column],
                yerr=None if error is None else 1.96 * error,
                marker="o",
                capsize=3,
                label=label,
            )
        grid = np.linspace(0.0, float(frame["target_t"].max()), 100)
        axis.plot(
            grid,
            stats.norm.cdf(1.959964 - grid) - stats.norm.cdf(-1.959964 - grid),
            linestyle=":",
            color="grey",
            label="Wald, theoretical",
        )
        axis.axhline(0.95, linestyle="--", linewidth=1.0, color="black")
        axis.set_xlabel("Calibrated bias-to-standard-error ratio $t$")
        axis.set_ylabel("Coverage probability")
        axis.set_ylim(0.0, 1.02)
        axis.set_title(f"Coverage across the bias family (n={sample_size})")
        axis.legend()
        figure.tight_layout()
        figure.savefig(FIGURE_DIR / f"e4_coverage_n{sample_size}.pdf", bbox_inches="tight")
        plt.show()

## 8. E5: how much rests on the reference allowance?

Two things are varied: the reference (`truth`, `correct`, `misspecified`, and `min`, the minimum bound over the two estimated ones) and the scale $\rho$ applied to the honest allowance. Setting $\rho = 0$ removes the allowance entirely and shows how much of the guarantee it carries.

The `misspecified` reference keeps the *same allowance formula* while dropping terms from both of its nuisance models, so its stated $b_r$ no longer bounds its drift. The `correct` reference includes the hidden direction in both nuisances and stays valid as $t$ grows; without that, every candidate bound would fail at once and the sweep would measure nothing.

Watch what the minimum bound of Proposition `several_references` does: it is valid only when *every* reference in the set is valid, so admitting one invalid reference damages the combined bound. The pairwise check is the intended defence, and the table below reports how often it actually fires.

In [ ]:
save(report.reference_robustness_table(tables), "e5_reference_robustness")

In [ ]:
save(report.reference_check_table(tables), "e5_reference_check")

## 9. E6: the price of bias awareness

`bound_to_se` is $\widehat U_{\widehat a} / \widehat{\mathrm{se}}_{\widehat a}$ and `bias_aware_over_wald` is the ratio of interval lengths. Corollary `oa:bias_aware_length` says the second tends to one as the first tends to zero, so these two columns should move together.

In [ ]:
save(report.interval_length_table(tables), "e6_interval_length")

## 10. Output files

Parquet results are written under `notebooks/experiments/results/reference_selection/<tier>/` and are not tracked by git. The CSV tables and PDF figures the manuscript reads go to the `tables` and `figures` subdirectories.

One caveat when transcribing numbers. The `bias_aware_pooled` interval is computed but has **no supporting theorem**: it applies the single-split bounded-normal-mean critical value to a cross-fitted estimate and a pooled standard error. `conservative_cf` is the cross-fitted interval the manuscript actually proves, and `bias_aware_split` is the single-split interval Theorem `uniform_selected_inference` covers. Compare the lengths of the three before deciding whether a cross-fitted bias-aware theorem is worth stating.